<a href="https://colab.research.google.com/github/Ignite-0/Fake-News-Detection-using-Transformers/blob/main/fake_news_detection_WELFakeDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Required Libraries
!pip install transformers datasets accelerate -q
!pip install scikit-learn matplotlib seaborn kagglehub -q

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import kagglehub
from collections import Counter
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    auc
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn

# Setup
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Using device: {device}")

# Set seeds
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(" All libraries imported!")


In [ ]:
# Download Dataset (WELFake)
print("\n" + "=" * 80)
print("=" * 80)
# Download the  dataset
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")
print(f" Dataset downloaded to: {path}")

# List files
import os
print("\n Files in dataset:")
for file in os.listdir(path):
    print(f"   - {file}")

In [ ]:
# Load and Analyze Dataset
print("\n" + "=" * 80)
print("LOADING WELFAKE DATASET")
print("=" * 80)

# Load the dataset
csv_file = os.path.join(path, 'WELFake_Dataset.csv')
df = pd.read_csv(csv_file)

print(f" Dataset loaded: {len(df):,} articles")
print(f"\n Columns: {df.columns.tolist()}")
print(f"\n First few rows:")
print(df.head())

# Check label distribution
print(f"\n Label Distribution:")
print(df['label'].value_counts())
print(f"\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

# Check for the label column
if 'label' in df.columns:
    print(f"\n Label column found!")
    print(f"   Unique labels: {df['label'].unique()}")
else:
    print(f"\n No 'label' column. Columns: {df.columns.tolist()}")


In [ ]:
# Data Preprocessing
# ============================================================================
print("\n" + "=" * 80)
print("DATA PREPROCESSING")
print("=" * 80)

# Combine title and text
if 'title' in df.columns and 'text' in df.columns:
    df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
elif 'text' in df.columns:
    df['full_text'] = df['text'].fillna('')
else:
    # Try to find the text column
    text_col = [col for col in df.columns if 'text' in col.lower()][0]
    df['full_text'] = df[text_col].fillna('')

print(f" Text column created")

# Clean text
def advanced_clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^\w\s.,!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['full_text'].apply(advanced_clean_text)

# Remove short texts
df = df[df['cleaned_text'].str.len() > 100].reset_index(drop=True)

print(f" Dataset after cleaning: {len(df):,} articles")

# Standardize labels to 0/1
# WELFake uses 0=real, 1=fake (opposite of what we had!)
if df['label'].dtype == 'int64':
    # Already numeric
    df['label_binary'] = df['label']
else:
    # Map string labels
    label_map = {
        'fake': 1, 'Fake': 1, 'FAKE': 1,
        'real': 0, 'Real': 0, 'REAL': 0,
        'true': 0, 'True': 0, 'TRUE': 0,
        'false': 1, 'False': 1, 'FALSE': 1,
        1: 1, 0: 0, '1': 1, '0': 0
    }
    df['label_binary'] = df['label'].map(label_map)

# Drop any unmapped labels
df = df.dropna(subset=['label_binary']).reset_index(drop=True)
df['label_binary'] = df['label_binary'].astype(int)

print(f"\n Final label distribution:")
print(f"   REAL (0): {(df['label_binary'] == 0).sum():,} ({(df['label_binary'] == 0).sum()/len(df)*100:.1f}%)")
print(f"   FAKE (1): {(df['label_binary'] == 1).sum():,} ({(df['label_binary'] == 1).sum()/len(df)*100:.1f}%)")

# Check if balanced
fake_pct = (df['label_binary'] == 1).sum() / len(df)
if 0.4 <= fake_pct <= 0.6:
    print(f"    Dataset is BALANCED!")
else:
    print(f"    Dataset is IMBALANCED - will use class weights")


In [ ]:
#  Train-Val-Test Split with Stratification
# ============================================================================
print("\n" + "=" * 80)
print("TRAIN-VALIDATION-TEST SPLIT (60-20-20)")
print("=" * 80)

# First split: 80-20 (train+val vs test)
train_val_df, test_df = train_test_split(
    df[['cleaned_text', 'label_binary']],
    test_size=0.20,
    random_state=42,
    stratify=df['label_binary']
)

# Second split: 75-25 of train+val ( 60-20 of total)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=42,
    stratify=train_val_df['label_binary']
)

print(f" Data split complete:")
print(f"   Training:   {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Validation: {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test:       {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

# Verify stratification
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    fake = (split_df['label_binary'] == 1).sum()
    real = (split_df['label_binary'] == 0).sum()
    print(f"\n   {name:10} REAL: {real:,} ({real/len(split_df)*100:.1f}%), FAKE: {fake:,} ({fake/len(split_df)*100:.1f}%)")


In [ ]:
# Calculate Class Weights
# ============================================================================
print("\n" + "=" * 80)
print("CALCULATING CLASS WEIGHTS")
print("=" * 80)

# Compute class weights to handle any imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_binary']),
    y=train_df['label_binary']
)

class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

print(f" Class weights calculated:")
print(f"   REAL (0): {class_weights[0]:.4f}")
print(f"   FAKE (1): {class_weights[1]:.4f}")

# Convert to torch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


In [ ]:
# Load RoBERTa Model
# ============================================================================
print("\n" + "=" * 80)
print("LOADING ROBERTA MODEL")
print("=" * 80)

model_name = "roberta-base"
print(f" Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type="single_label_classification"
)

model.to(device)
print(f" Model loaded on {device}")
print(f"   Parameters: {model.num_parameters():,}")

In [ ]:
# Custom Trainer with Weighted Loss
# ============================================================================
print("\n" + "=" * 80)
print("CREATING CUSTOM TRAINER WITH CLASS WEIGHTS")
print("=" * 80)

class WeightedTrainer(Trainer):
    """Custom trainer with weighted cross-entropy loss"""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Weighted cross-entropy loss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

print(" Custom trainer class created with weighted loss")

In [ ]:
# Tokenization
# ============================================================================
print("\n" + "=" * 80)
print("TOKENIZING DATASETS")
print("=" * 80)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512,
        return_attention_mask=True
    )

# Create datasets
train_dataset = Dataset.from_pandas(
    train_df.rename(columns={'cleaned_text': 'text', 'label_binary': 'labels'})
)
val_dataset = Dataset.from_pandas(
    val_df.rename(columns={'cleaned_text': 'text', 'label_binary': 'labels'})
)
test_dataset = Dataset.from_pandas(
    test_df.rename(columns={'cleaned_text': 'text', 'label_binary': 'labels'})
)

# Tokenize
print(" Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(" Tokenization complete!")

In [ ]:
# Define Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', pos_label=1, zero_division=0
    )

    # Per-class metrics
    precision_per, recall_per, f1_per, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    acc = accuracy_score(labels, preds)

    # PR-AUC
    probs = torch.nn.functional.softmax(torch.tensor(predictions), dim=-1).numpy()
    precision_curve, recall_curve, _ = precision_recall_curve(labels, probs[:, 1], pos_label=1)
    pr_auc = auc(recall_curve, precision_curve)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f1_real': f1_per[0],
        'f1_fake': f1_per[1],
        'pr_auc': pr_auc
    }

In [ ]:
# Training Configuration
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    gradient_accumulation_steps=1,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none"
)

print(" Training configuration set")

In [ ]:
# Initialize Weighted Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print(" Weighted trainer initialized!")

In [ ]:
# Train Model
print("\n" + "=" * 80)
print(" TRAINING MODEL WITH CLASS WEIGHTS")
print("=" * 80)

train_result = trainer.train()

print("\n Training complete!")
print(f"   Loss: {train_result.training_loss:.4f}")
print(f"   Time: {train_result.metrics['train_runtime']/60:.1f} minutes")

In [ ]:
# Evaluate on Test Set
print("\n" + "=" * 80)
print(" FINAL EVALUATION ON TEST SET")
print("=" * 80)

test_predictions = trainer.predict(test_dataset)
test_results = test_predictions.metrics

print("\n TEST SET RESULTS:")
print("=" * 80)
for key, value in test_results.items():
    if isinstance(value, float) and 'test' in key:
        metric_name = key.replace('test_', '').upper()
        print(f"   {metric_name:20} {value:.4f}")

# Get predictions
pred_probs = torch.nn.functional.softmax(torch.tensor(test_predictions.predictions), dim=-1).numpy()
pred_labels = np.argmax(test_predictions.predictions, axis=1)
true_labels = test_predictions.label_ids

In [ ]:
# Detailed Analysis
# ============================================================================
print("\n" + "=" * 80)
print(" VERIFICATION - DOES IT ACTUALLY WORK?")
print("=" * 80)

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

print("\nConfusion Matrix:")
print(f"                Predicted")
print(f"                REAL    FAKE")
print(f"Actual REAL     {cm[0,0]:<6}  {cm[0,1]:<6}")
print(f"Actual FAKE     {cm[1,0]:<6}  {cm[1,1]:<6}")

# Per-class accuracy
real_correct = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
fake_correct = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print(f"\n PER-CLASS ACCURACY (THE REAL TEST!):")
print(f"   REAL news correctly identified: {real_correct:.1%} ({cm[0,0]}/{cm[0,0]+cm[0,1]})")
print(f"   FAKE news correctly identified: {fake_correct:.1%} ({cm[1,1]}/{cm[1,0]+cm[1,1]})")

if real_correct > 0.80 and fake_correct > 0.80:
    print(f"\n    MODEL IS ACTUALLY USEFUL! Both classes detected well!")
elif real_correct < 0.50 or fake_correct < 0.50:
    print(f"\n    WARNING: Model is biased towards one class!")
else:
    print(f"\n    Model works, but could be better")

# Classification report
print("\n DETAILED CLASSIFICATION REPORT:")
report = classification_report(
    true_labels,
    pred_labels,
    target_names=['REAL', 'FAKE'],
    digits=4
)
print(report)

In [ ]:
# Visual Analysis
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay
cmd = ConfusionMatrixDisplay(cm, display_labels=['REAL', 'FAKE'])
cmd.plot(ax=axes[0, 0], cmap='Blues')
axes[0, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# PR curve
precision_curve, recall_curve, _ = precision_recall_curve(true_labels, pred_probs[:, 1], pos_label=1)
pr_auc_score = auc(recall_curve, precision_curve)

axes[0, 1].plot(recall_curve, precision_curve, linewidth=2, label=f'PR-AUC = {pr_auc_score:.4f}')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Prediction distribution
axes[1, 0].hist(pred_probs[true_labels==0, 1], bins=30, alpha=0.7, label='REAL (True)', color='#2ecc71')
axes[1, 0].hist(pred_probs[true_labels==1, 1], bins=30, alpha=0.7, label='FAKE (True)', color='#e74c3c')
axes[1, 0].set_xlabel('Predicted Probability (FAKE)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Prediction Distribution', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].axvline(x=0.5, color='black', linestyle='--', label='Threshold')

# Per-class F1
f1_scores = [test_results['test_f1_real'], test_results['test_f1_fake']]
axes[1, 1].bar(['REAL', 'FAKE'], f1_scores, color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_ylim([0, 1])
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].set_title('Per-Class F1-Scores', fontsize=14, fontweight='bold')
axes[1, 1].axhline(y=0.8, color='orange', linestyle='--', label='Good (0.8)')
axes[1, 1].legend()

for i, v in enumerate(f1_scores):
    axes[1, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Test with Real Examples
# ============================================================================
print("\n" + "=" * 80)
print(" TESTING WITH REAL EXAMPLES")
print("=" * 80)

def predict_news(text):
    """Predict if news is fake or real"""
    cleaned = advanced_clean_text(text)
    inputs = tokenizer(cleaned, return_tensors='pt', truncation=True, max_length=512, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_class].item()

    return {
        'label': 'FAKE' if pred_class == 1 else 'REAL',
        'confidence': confidence,
        'real_prob': probs[0][0].item(),
        'fake_prob': probs[0][1].item()
    }

# Test with diverse examples
test_cases = [
    # Should be REAL
    ("The Federal Reserve announced today that interest rates will remain unchanged following their monthly meeting.", "REAL"),
    ("Stock market closes higher as investors react to positive employment data released this morning.", "REAL"),
    ("The Supreme Court ruled on a landmark case regarding constitutional rights this morning in Washington.", "REAL"),

    # Should be FAKE
    ("Breaking: Scientists discover aliens living on Mars! NASA confirms shocking findings!", "FAKE"),
    ("You won't believe this one weird trick that doctors don't want you to know about!", "FAKE"),
    ("SHOCKING: Government hiding the truth about vaccines! Click here to learn the real story!", "FAKE"),
]

correct_predictions = 0
print("\n Testing diverse examples:\n")

for i, (text, expected) in enumerate(test_cases, 1):
    result = predict_news(text)
    is_correct = result['label'] == expected
    correct_predictions += is_correct

    emoji = "✅" if is_correct else "❌"
    status = "CORRECT" if is_correct else "WRONG"

    print(f"{i}. {emoji} {status} - Predicted: {result['label']} (Expected: {expected})")
    print(f"   Confidence: {result['confidence']:.1%}")
    print(f"   Text: {text[:80]}...")
    print(f"   Probs: REAL={result['real_prob']:.1%}, FAKE={result['fake_prob']:.1%}\n")

accuracy = correct_predictions / len(test_cases)
print(f"{'='*80}")
print(f" Manual Test Accuracy: {accuracy:.1%} ({correct_predictions}/{len(test_cases)})")
print(f"{'='*80}")

if accuracy >= 0.8:
    print("\n MODEL WORKS WELL ON DIVERSE EXAMPLES!")
else:
    print("\n Model needs improvement on diverse examples")